# 03a - RM-a: full fine-tuning IndoBERT

Baseline penelitian. Seluruh 109 juta parameter IndoBERT diperbarui, sehingga
skenario ini yang paling mahal sekaligus menjadi pembanding performa untuk dua
strategi ringan.

Notebook ini menjalankan SATU konfigurasi: baseline kanonik IndoNLU/Wilie (2020)
`lr=2e-5, epochs=5, batch=16, warmup=0,1, wd=0,01`. Eksplorasi hyperparameter
ada di `04_tuning_campaign.ipynb`.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [1]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

/workspace/indobert-with-rac/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-14 16:45:27,853 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device        : cuda
encoder       : indobenchmark/indobert-base-p2
keluaran      : /workspace/indobert-with-rac/outputs/baseline
train/val/test: [6588, 1402, 1405]


## 1. Catat konteks hardware

In [2]:
hardware = runner.write_hardware()
for key, value in hardware.items():
    print(f"  {key:16s}: {value}")

  gpu             : NVIDIA GeForce RTX 3090
  cuda_available  : True
  cuda_version    : 13.0
  torch           : 2.12.1+cu130
  transformers    : 5.12.1
  vram_total_mb   : 24124
  recorded_at     : 2026-09-14 16:45:29
  nvidia_smi      : NVIDIA GeForce RTX 3090, 580.178.04, 24576 MiB


Angka efisiensi hanya bisa ditafsirkan bersama konteks ini, dan hanya sebanding
bila seluruh skenario diukur pada hardware dan sesi yang sama.

## 2. Konfigurasi

In [3]:
from src.models.schemas import RMAConfig

config = RMAConfig()
print(config.model_dump())
print(f"\nbatch efektif {config.batch} dicapai lewat micro-batch "
      f"{config.effective_micro_batch} x akumulasi {config.grad_accum}")

{'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}

batch efektif 16 dicapai lewat micro-batch 8 x akumulasi 2


Akumulasi gradien membuat RUMUS gradiennya ekuivalen dengan batch besar (BERT
memakai LayerNorm, bukan BatchNorm, dan loss dibagi jumlah akumulasi), tetapi
RUN-nya tidak identik: `DataLoader` dengan ukuran batch berbeda mengonsumsi RNG
secara berbeda, sehingga mask dropout dan komposisi tiap batch ikut berubah.

Terukur di kampanye ini: `micro_batch` efektif 8 versus 16 pada konfigurasi yang
sama persis memberi val F1-macro 0,974873 versus 0,977266. Karena itu
`micro_batch` harus dikunci untuk seluruh sel satu grid dan disebutkan di Bab 4
sebagai bagian konfigurasi, bukan diperlakukan sebagai knob memori bebas.

## 3. Jalankan

In [4]:
row = runner.run(
    "rma",
    config.model_dump(),
    note="baseline kanonik IndoNLU/Wilie 2020, titik acuan seluruh grid",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro    : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi     : {row['val_f1_judi']:.4f}")
print(f"  epoch terbaik   : {row['best_epoch']} dari {row['epochs']}")
print(f"  waktu latih     : {row['train_time_s']:.1f} s")
print(f"  peak GPU memory : {row['peak_mem_mb']:.0f} MB")
print(f"  trainable params: {row['trainable_params']:,}")

2026-09-14 16:45:29,297 | INFO     | src.services.campaign | [rma] RUN #1 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 55882.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 16:45:45,331 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-14 16:46:11,764 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9586
2026-09-14 16:46:38,000 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9682
2026-09-14 16:47:04,240 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9762
2026-09-14 16:47:29,908 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9762
2026-09-14 16:47:55,337 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9784
2026-09-14 16:47:55,813 | INFO     | src.services.run_log | Juara baru untuk rma: val F1-macro 0.9784 (sebelumnya -1.0000)
2026-09-14 16:47:57,723 | INFO     | src.services.campaign | [rma] RUN #1 val F1-macro 0.9784 (epoch terbaik 5)
run #1
  val F1-macro    : 0.9784
  val F1 judi     : 0.9646
  epoch terbaik   : 5 dari 5
  waktu latih     : 130.1 s
  peak GPU memory : 2334 MB
  trainable params: 109,485,314

## 4. Kurva per epoch

In [5]:
import pandas as pd

history = pd.read_csv(OUT_DIR / "history" / "rma_history.csv")
history[history["run_id"] == row["run_id"]]

,run_id,epoch,train_loss,val_f1_macro,val_acc,val_f1_judi,val_precision_judi,val_recall_judi
0,1,1,0.220769,0.958599,0.974322,0.933086,0.893238,0.976654
1,1,2,0.076079,0.968175,0.980742,0.948177,0.935606,0.961089
2,1,3,0.040232,0.976250,0.985735,0.961240,0.957529,0.964981
3,1,4,0.019780,0.976250,0.985735,0.961240,0.957529,0.964981
4,1,5,0.010262,0.978364,0.987161,0.964567,0.976096,0.953307


`overfit_signal` di baris riwayat bernilai True bila epoch terbaik bukan epoch
terakhir, yaitu tanda bahwa menambah epoch justru memperburuk validasi.

Figur kurva tersimpan di `outputs/baseline/figures/rma_run{id}_curve.png`.

In [ ]:
## Ringkasan

Angka di atas adalah baseline satu konfigurasi, bukan hasil final. Konfigurasi
final ditentukan lewat kampanye di `04_tuning_campaign.ipynb`, dan angka test
baru dibuka sekali di `05_final_benchmark.ipynb`.

Lanjut ke `03b_rmb_frozen.ipynb`.